# Polarized Trees — Benchmark (hyperparameter search)

This is the **second** of four notebooks in the synthetic workflow:

1. `datasetdemo.ipynb` — generate the fixed synthetic corpora;
2. **this notebook** — search the hyperparameters and save the results, including the top 20 configurations;
3. `resultsexploration.ipynb` — explore the saved results: the top 20, which formulations and hyperparameters matter, recovery by corpus and by k;
4. `pegcomparison.ipynb` — take settings from the top 20, keep everything fixed except the PEG formulation, and compare the formulations (recovery, inference, tree shape).

It is a simple example of how to use `PolarizedTreesBenchmark`, and it is also the reproducibility code for the synthetic benchmark reported in the paper.

**fixed synthetic data → hyperparameter search → saved results (top 20)**

The benchmark module performs the actual model selection. The notebook does not manually rank configurations or reconstruct the selected model.

> **Important:** the synthetic corpora must be generated first with `datasetdemo.ipynb`. This notebook reuses those fixed datasets so every configuration is evaluated on exactly the same data.

## What is fixed and what can be changed?

This notebook uses the **default benchmark settings provided by `polartox.benchmark`**. These defaults give a ready-to-run starting point, but they are not required.

The main benchmark settings can be changed:

- **search space** — pass a dictionary defining the values to consider for each hyperparameter;
- **strategy** — `"full"` evaluates every configuration, while `"random"` samples configurations;
- **number of runs** — controls how many configurations are sampled with random search;
- **seed** — makes random search reproducible;
- **metrics** — choose which recovery metrics to calculate;
- **selection metric** — choose which metric determines the best configuration;
- **selection direction** — choose `"max"` or `"min"`;
- **pipeline settings** — dimensions, scale, and any fixed pipeline parameters.

For this notebook, the default search space, metrics, and selection metric are imported directly from the package. You can replace `DEFAULT_SEARCH_SPACE` with your own dictionary without changing the benchmark code.

Ground truth is required for recovery-based model selection. After the best configuration is selected, the resulting pipeline can be run **without ground truth**, which is the normal inference setting for real annotation data.


## 1. Imports and experiment settings

Only a small amount of configuration is needed here. The actual search, evaluation, ranking, and selection are delegated to `PolarizedTreesBenchmark`.


In [1]:
import sys
from pathlib import Path

import pandas as pd

from polartox.benchmark import (
    PolarizedTreesBenchmark,
    DEFAULT_SEARCH_SPACE,
    DEFAULT_METRICS,
    DEFAULT_SELECTION_METRIC,
)
from polartox.pipeline import PolarizedTreesPipeline


# Locate the repository root.
CWD = Path.cwd().resolve()

if (CWD / "benchmark_config.py").exists():
    PROJECT_ROOT = CWD
elif (CWD.parent / "benchmark_config.py").exists():
    PROJECT_ROOT = CWD.parent
else:
    raise FileNotFoundError(
        "Could not locate project root. "
        "Run this notebook from the notebooks directory "
        "or the project root."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


from benchmark_config import (
    BENCHMARK_CORPORA,
    DIMS,
    RESULTS_DIR,
    SCALE,
    check_polartox,
    load_benchmark_corpora,
)

# Which polartox and which library versions produce these results.
check_polartox()


RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Paper settings.
N_RUNS = 800
SEED = 0

print("Project root:", PROJECT_ROOT)
print("Benchmark corpora:", BENCHMARK_CORPORA)
print("Random configurations:", N_RUNS)
print("Seed:", SEED)

polartox 0.8.1 imported from C:\Users\Sokratis\OneDrive\Υπολογιστής\ερευνα\HumanModelAlign\polarizedtrees\polartox\__init__.py
PEG formulations: ('max', 'weighted', 'min', 'mean', 'harmonic')
numpy 2.4.6
pandas 3.1.1
ndfu 0.9.3
scikit-learn 1.9.1
Project root: C:\Users\Sokratis\OneDrive\Υπολογιστής\ερευνα\HumanModelAlign\polarizedtrees\benchmarks
Benchmark corpora: ['A_default', 'B_weak_signal', 'C_deep', 'D_strong_signal', 'E_very_weak', 'F_shallow', 'G_uniform_depth', 'H_deep_weak', 'I_deep_strong', 'J_shallow_weak']
Random configurations: 800
Seed: 0


## 2. Load the fixed synthetic datasets

The first notebook generates these datasets once and stores both the annotations and the per-text ground truth.

Ground truth tells us which SCD dimensions actually generated the polarization. That makes recovery metrics possible and allows the benchmark to perform model selection.

The unseen inference corpus is not loaded here: it is not used for selecting the configuration, and `pegcomparison.ipynb` uses it.

In [2]:
corpora = load_benchmark_corpora()

for name, (dataset, ground_truth) in corpora.items():
    print(
        f"{name}: "
        f"{dataset.shape[0]:,} annotations | "
        f"{dataset['text_id'].nunique()} texts | "
        f"{len(ground_truth)} ground-truth entries"
    )

A_default: 162,000 annotations | 100 texts | 100 ground-truth entries
B_weak_signal: 162,000 annotations | 100 texts | 100 ground-truth entries
C_deep: 162,000 annotations | 100 texts | 100 ground-truth entries
D_strong_signal: 162,000 annotations | 100 texts | 100 ground-truth entries
E_very_weak: 162,000 annotations | 100 texts | 100 ground-truth entries
F_shallow: 162,000 annotations | 100 texts | 100 ground-truth entries
G_uniform_depth: 162,000 annotations | 100 texts | 100 ground-truth entries
H_deep_weak: 162,000 annotations | 100 texts | 100 ground-truth entries
I_deep_strong: 162,000 annotations | 100 texts | 100 ground-truth entries
J_shallow_weak: 162,000 annotations | 100 texts | 100 ground-truth entries


## 3. Prepare the model-selection data

`PolarizedTreesBenchmark` works on one annotation dataset. We therefore combine the ten corpora (A to J) into one development corpus.

The text IDs are shifted so that the ten corpora remain completely distinct. The ten corpora contain the same number of texts, so their combined mean gives each corpus equal weight.

This is only a data-preparation step; it does not change the annotations or their ground truth.


In [3]:
text_counts = {
    name: dataset["text_id"].nunique()
    for name, (dataset, _) in corpora.items()
}

if len(set(text_counts.values())) != 1:
    raise ValueError(
        "The corpora must contain the same number of texts "
        "for equal-weight mean Jaccard."
    )


benchmark_parts = []
benchmark_ground_truth = {}
text_groups = {}

offset = 0

for corpus_name in BENCHMARK_CORPORA:
    dataset, ground_truth = corpora[corpus_name]

    part = dataset.copy()

    text_ids = sorted(
        part["text_id"].unique()
    )

    id_map = {
        old_id: offset + i
        for i, old_id in enumerate(text_ids)
    }

    part["text_id"] = part["text_id"].map(id_map)

    benchmark_parts.append(part)

    for old_id, config in ground_truth.items():
        benchmark_ground_truth[
            id_map[old_id]
        ] = config

    for new_id in id_map.values():
        text_groups[new_id] = corpus_name

    offset += len(text_ids)


benchmark_dataset = pd.concat(
    benchmark_parts,
    ignore_index=True,
)

print("Combined annotations:", benchmark_dataset.shape)
print("Combined texts:", benchmark_dataset["text_id"].nunique())
print("Ground-truth entries:", len(benchmark_ground_truth))


Combined annotations: (1620000, 8)
Combined texts: 1000
Ground-truth entries: 1000


## 4. Define the paper's search space

For the paper, we consider the full valid search space of 3,240 configurations.

The space is the Cartesian product of the values below: 648 settings of the other hyperparameters times the five PEG formulations (`max`, `weighted`, `min`, `mean`, `harmonic`).

For a smaller demonstration, this cell can be replaced by a small custom search space. The benchmark API supports both forms.


In [4]:
SEARCH_SPACE = DEFAULT_SEARCH_SPACE.copy()

print("Search space:")
for parameter, values in SEARCH_SPACE.items():
    print(f"  {parameter}: {values}")

n_configurations = 1
for values in SEARCH_SPACE.values():
    n_configurations *= len(values)

print("\nTotal possible configurations:", n_configurations)


Search space:
  theta_filter: [0.2, 0.3, 0.4]
  min_size_frac: [0.02, 0.03, 0.05]
  max_depth: [4, 6, 8]
  variant: ['max', 'weighted', 'min', 'mean', 'harmonic']
  h: [0.05, 0.1, 0.15, 0.2]
  relative_h: [False, True]
  theta_stop: [0.05, 0.1, 0.15]

Total possible configurations: 3240


## 5. Create the base Polarized Trees pipeline

The user provides **one pipeline object** to the benchmark.

The benchmark uses its settings as the base and creates candidate pipelines internally while testing configurations. After model selection, the selected pipeline is available directly through `benchmark.get_best_pipeline()`.

The user therefore does not need to manually reconstruct the winning pipeline.


In [5]:
# theta_filter, h and max_depth are required by the pipeline; the
# benchmark overrides all of them with each searched configuration,
# so these are placeholders.
pipeline = PolarizedTreesPipeline(
    dims=DIMS,
    scale=SCALE,
    theta_filter=0.3,
    h=0.10,
    max_depth=6,
)

print(pipeline)


## 6. Run model selection

This is the main benchmark step.

The benchmark:

1. generates the configurations from the supplied search-space dictionary;
2. evaluates the requested configurations on the same annotations and ground truth;
3. computes the requested recovery metrics;
4. ranks the configurations using the selected metric;
5. stores the best configuration and the corresponding pipeline.

Here we use random search so that the same notebook can also demonstrate the `random` strategy. For a small search space, `strategy="full"` is useful because it evaluates every possible configuration.

The defaults are imported from the package, but all of these choices can be changed.


In [6]:
benchmark = PolarizedTreesBenchmark(
    pipeline=pipeline,
    annotations=benchmark_dataset,
    ground_truth=benchmark_ground_truth,

    # Corpus of every text: metrics and statistics are computed per
    # corpus and then averaged over corpora.
    text_groups=text_groups,

    # Paper search.
    search_space=SEARCH_SPACE,
    strategy="random",
    n_runs=N_RUNS,
    seed=SEED,

    # Recovery metrics available in the synthetic setting.
    metrics=DEFAULT_METRICS,

    # Paper selection criterion.
    selection_metric=DEFAULT_SELECTION_METRIC,

    # Keep the highest score.
    selection_direction="max",

    top_k=20,
    verbose=True,

    # Save progress every 50 configurations; re-running this cell
    # after a crash resumes from the last checkpoint.
    checkpoint_dir=RESULTS_DIR / "checkpoint",
    checkpoint_every=50,

    # Evaluate configurations in parallel worker processes. Each worker
    # holds its own copy of the data (~150 MB); lower this if memory is tight.
    n_jobs=6,
)

benchmark.run()


[1/800] jaccard=0.8162
[2/800] jaccard=0.8006


[3/800] jaccard=0.5303


[4/800] jaccard=0.8104


[5/800] jaccard=0.8435
[6/800] jaccard=0.8211


[7/800] jaccard=0.8079


[8/800] jaccard=0.8547
[9/800] jaccard=0.8055


... (556 outputs left out)

[796/800] jaccard=0.5704


[797/800] jaccard=0.7874
[798/800] jaccard=0.5643


[799/800] jaccard=0.8186


[800/800] jaccard=0.8211


## 7. Save the benchmark results

The search is finished. Everything is written to `benchmark_results/`:

- `benchmark_configuration_summary.csv` — one row per configuration (averaged over corpora);
- `benchmark_results.csv` — one row per configuration and corpus;
- `benchmark_text_results.csv` — the raw per-text recovery values;
- `benchmark_report.json` — settings, search space, best configuration and top configurations;
- `top_configurations.csv` — the top 20, which `pegcomparison.ipynb` opens as a table.

Looking into these results (which formulations and hyperparameters matter, recovery by corpus and by k) is the next notebook, `resultsexploration.ipynb`.

In [7]:
benchmark.save_results(RESULTS_DIR / "benchmark_configuration_summary.csv")
benchmark.save_group_results(RESULTS_DIR / "benchmark_results.csv")
benchmark.save_text_results(RESULTS_DIR / "benchmark_text_results.csv")
benchmark.save_report(RESULTS_DIR / "benchmark_report.json")
benchmark.get_top_configs().to_csv(RESULTS_DIR / "top_configurations.csv", index=False)

## 8. The selected configuration

Nothing is selected manually here. These values are returned by the benchmark after ranking all evaluated configurations. The rest of the results is explored in `resultsexploration.ipynb`.

In [8]:
print("Best configuration:")
print(benchmark.get_best_config())
print()
print("Best mean Jaccard:")
print(benchmark.get_best_score())

Best configuration:
{'theta_filter': 0.3, 'min_size_frac': 0.02, 'max_depth': 4, 'variant': 'harmonic', 'h': 0.2, 'relative_h': True, 'theta_stop': 0.1}

Best mean Jaccard:
0.8885713168082455


## 9. Adapting this notebook to another experiment

The notebook uses the package defaults as a convenient starting point. The benchmark itself is deliberately configurable.

The main things to change are:

```text
search_space        → a dictionary of hyperparameter values to search
strategy            → "full" or "random"
n_runs              → number sampled for random search
seed                → reproducible random sampling
metrics             → recovery metrics to compute
selection_metric    → metric used to choose the winner
selection_direction → "max" or "min"
pipeline            → fixed dimensions, scale, and other settings
annotations         → your annotation dataset
ground_truth        → required for recovery-based model selection
```

For example, a custom search can be as simple as:

```python
SEARCH_SPACE = {
    "relative_h": [True],
    "h": [0.05, 0.10, 0.15],
    "max_depth": [4, 6],
}
```

You then pass that dictionary directly to `PolarizedTreesBenchmark`.

If you already have an annotation dataset, you can use it directly as long as it follows the expected annotation format. Synthetic data is only needed when you want known ground truth for objective recovery evaluation.

If you do not have ground truth, the benchmark cannot use recovery metrics for model selection. In that case, use `PolarizedTreesPipeline` directly in inference mode and interpret F/C/P and diagnostics.

## Next steps

1. Run `datasetdemo.ipynb` to generate the fixed synthetic datasets.
2. Run this notebook to search the hyperparameters and save the results.
3. Run `resultsexploration.ipynb` to explore the results: the top 20, the effect of each hyperparameter, recovery by corpus and by k.
4. Run `pegcomparison.ipynb` to compare the PEG formulations on the best configurations.
5. For another experiment, change the search-space dictionary and/or benchmark settings and rerun.